# Preliminaries

In this notebook, we are going to simulate a 2-Qubit-Coupler and drive it with a new gate scheme. You will:

(1) derive the effective Hamiltonian of the system,

(2) extract its effective description, which allows you to predict the parameters of the gate protocol

(3) numerically simulate the gate and check its fidelity

Before we start: clone the github project with the needed python packages and data.

We propose: run the following cell to clone it and move into the repo folder

In [1]:
pip install numpy scipy matplotlib sympy xarray h5netcdf qutip tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
from Floquet_perturbation_theory import *
from qutip import *
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp
import xarray as xr
from scipy.optimize import fsolve
from tqdm.notebook import tqdm
from helper_function import *
from utils import *

from sympy.physics.quantum import Dagger
from sympy.physics.quantum.boson import BosonOp
from sympy.physics.vector import dynamicsymbols, init_vprinting

# Initial Numerical Implementation of the System

After knowing the the explicit form of the Hamiltonian, let's implement it for numerical simulations. To eventually perform a gate, we need to know: (a) the drive frequency and (b) the gate time (corresponding to the Rabi rate of the swap).
To predict these values precisely, we will numerically compute an *effective Hamiltonian* via perturbation theory. To make use of the framework that follows the Hamiltonian needs to be split as follows:
$$
\begin{gathered}
    H (t) = \sum_i E_i |i\rangle\langle i| + V(t)\\
\end{gathered}
$$
where $ \sum_i E_i |i\rangle\langle i|$ is completely diagonal (describing the system without interactions) and all interactions and time depency needs to be in the perturbative part $V(t)=V_0+V_1e^{i\omega_d t}+h.c.$.

First, we construct the time-independent part $E$:

In [3]:
# we use the typical operators to indicate the non-zero matrix elements between the states of interest.
dim_q1 = 3
dim_q2 = 3
dim_c = 3

w1, w2, wc, alpha1, alpha2, alphac, g1c, g2c, g12, c1, c2 = sp.symbols(
    "\\omega_1, \\omega_2, \\omega_c, \\alpha_1, \\alpha_2, \\alpha_c, g_{1c}, g_{2c}, g_{12}, c_{1}, c_{2}", 
    real=True,
)

a_q1 = tensor(destroy(dim_q1), qeye(dim_c), qeye(dim_q2))
a_q2 = tensor(qeye(dim_q1), qeye(dim_c), destroy(dim_q2))
a_qc = tensor(qeye(dim_q1), destroy(dim_c), qeye(dim_q2))

# sowohl die harmonische als auch die erste anharmonische Ordnung ist eine Funktion von n, also diagonal 
# a†a†aa = n(n-1) np.diag() liefert eine Liste der 27 Diagonaleinträge

The analytical analysis is done in the dressed basis, in which V0 is non existant and the only relevant fourier component is given by V1. 

In [4]:
# define a function to tell whether a certain V1 matrix element is existing
s = Simulation()

V0 = (g12 * ((a_q1 @ a_q2.dag()).full() + (a_q1.dag() @ a_q2).full() + (a_q1 @ a_q2).full() + (a_q1.dag() @ a_q2.dag()).full()) + 
      g1c * ((a_q1 @ a_qc.dag()).full() + (a_q1.dag() @ a_qc).full() + (a_q1 @ a_qc).full() + (a_q1.dag() @ a_qc.dag()).full()) +
      g2c * ((a_q2 @ a_qc.dag()).full() + (a_q2.dag() @ a_qc).full() + (a_q2 @ a_qc).full() + (a_q2.dag() @ a_qc.dag()).full()))
# V1 = c1*np.square((a_qc.dag() @ a_qc).full()) + c2*(a_qc.dag() @ a_qc).full()
V1 = c2 * (a_qc.dag() @ a_qc).full()

E_analytics, V0_analytics, V1_analytics = get_analytical_matrices(
    dim_q1, dim_c, dim_q2, V0, V1
)

# Testing the Heff Prediction

To enable the computing in stark shifts, we first determine two states involve in the dynamics.

In [5]:
wd = dynamicsymbols("\\omega_{d}")
dwd = wd.diff()
t = sp.symbols('t', real=True)

Test the analytical prediction for the effective Hamiltonian. 

In [6]:
rH = 3
Omega_01 = Heff_Floquet_summed(rH, s.state_a, s.state_b, wd, s.resonances, E_analytics, V1_analytics, V0_analytics, analytics=True, dwd=dwd, t=t)
simplified = sp.collect(Omega_01, wd)
len(simplified.args)
simplified

\bra{001}V_0{\ket{010}}*\bra{010}V_0{\ket{100}}*conjugate(\bra{010}V_1{\ket{010}})/(2*\omega_{d}(t)**2) + \bra{001}V_0{\ket{111}}*\bra{111}V_0{\ket{100}}*conjugate(\bra{111}V_1{\ket{111}})/((E_{\ket{001}} - E_{\ket{111}})*(E_{\ket{001}} - E_{\ket{111}} + \omega_{d}(t)))

## Implementation of time dependent parameters

Apart from the resonant frequency, another important quantity we are intetered in is the rabi rate, which determines the time for $\pi$ rotation or gate time of iSWAP. Let's first try with the summed in the first and second order processes.

In [7]:
t = sp.symbols("t", real=True)
popt_real = [ 1.59629893e+00, 1.25000077e+02, 3.74433146e+01, -5.60979110e-03]
popt_imag = [ 4.00236190e-02, 1.24975571e+02,  4.48865078e+01, -2.19686977e-02, 3.75630992e-03,  5.80637185e-05]
popt_freq = [2.10515838e-02, 1.25000139e+02, 2.66358933e+01, 4.43802950e+00]
A, t0, sigma, c = popt_real
amp_sym_real = (c + A*sp.exp(-(t-t0)**2/(2*sigma**2)))
A, t0, sigma, omega, phi, c = popt_imag
amp_sym = amp_sym_real + 1j*(c + A * sp.sin(omega*(t-t0)+phi) * sp.exp(-(t-t0)**2/(2*sigma**2)))
A, t0, sigma, c = popt_freq
freq_sym = c + A*sp.exp(-(t-t0)**2/(2*sigma**2))

rH = 3
Omega_01 = Heff_Floquet_summed(rH, s.state_a, s.state_b, freq_sym, s.resonances, s.E_array, 
                               s.V1_dressed_array * amp_sym/2, V0=None, analytics=True)

KeyboardInterrupt: 

In [ ]:
# simplify 
simplified = sp.collect(Omega_01, wd)

In [ ]:
display(simplified)

-0.0106205947176049*I*(5.80637185e-5 - 0.040023619*exp(-0.000248163764034759*(t - 124.975571)**2)*sin(0.0219686977*t - 2.74930684910389)) + 0.000347951228218184*(-0.0106205947176049*I*(5.80637185e-5 - 0.040023619*exp(-0.000248163764034759*(t - 124.975571)**2)*sin(0.0219686977*t - 2.74930684910389)) - 5.9579317723527e-5 + 0.0169536439836764*exp(-0.000356632920915851*(t - 125.000077)**2))*(-0.00377196034959184*I*(5.80637185e-5 - 0.040023619*exp(-0.000248163764034759*(t - 124.975571)**2)*sin(0.0219686977*t - 2.74930684910389)) + 2.11599095986932e-5 - 0.00602117627005588*exp(-0.000356632920915851*(t - 125.000077)**2))*(-0.00188598017479592*I*(5.80637185e-5 - 0.040023619*exp(-0.000248163764034759*(t - 124.975571)**2)*sin(0.0219686977*t - 2.74930684910389)) - 1.05799547993466e-5 + 0.00301058813502794*exp(-0.000356632920915851*(t - 125.000077)**2)) + 0.000459680278010994*(-0.0106205947176049*I*(5.80637185e-5 - 0.040023619*exp(-0.000248163764034759*(t - 124.975571)**2)*sin(0.0219686977*t - 2.7

From this anlaytical expression, we can tell that the population transferring is enabled by states in two paths by the 3rd order process.

- $|001, p=1\rangle \rightarrow |010, p=1\rangle \rightarrow |010, p=0\rangle \rightarrow |100,p=0\rangle$
- $|001, p=1\rangle \rightarrow |111, p=1\rangle \rightarrow |111, p=0\rangle \rightarrow |100,p=0\rangle$

The processes consist of parametric modulation of middle coupler frequency as well as pairwise swapping and creation.

In [ ]:
def resonant_condition(fre, amp, order, i, f, E0, V0, V1, analytics=False):
    fre = float(np.atleast_1d(fre)[0])
    delta_i = Heff_Floquet_summed(
        order,
        i,
        i,
        fre,
        resonances,  # E_i - wd = E_f
        E0,
        amp / 2 * V1,
        V0=V0,
        analytics=analytics,
    )
    delta_f = Heff_Floquet_summed(
        order,
        f,
        f,
        fre,
        resonances,  # E_i - wd = E_f
        E0,
        amp / 2 * V1,
        V0=V0,
        analytics=analytics,
    )
    diff = delta_f - delta_i
    return float(np.real(diff))

def rabi_rate(fre, amp, order, E0, V0, V1):
    i = state_b
    f = state_a
    fre = float(np.atleast_1d(fre)[0])
    omega_r = Heff_Floquet_summed(
        order, 
        i, 
        f,
        fre, 
        resonances,
        E0, 
        amp / 2 * V1,
        V0,
        analytics=False,
    )
    return omega_r

[**Exercise**: Now it is your turn: Construct a function called "rabi_rate", which returns the off-diagonal matrix elements in the effective Hamiltonian. You may look up the function "resonant_condition" we built before as a hint.]

# Numerical Search for Gate paramaters

So far, we have developed an analytical description in the product basis of the coupled transmon system. This gives a clear physical picture of the parametric interaction, where the dynamics arise from both static couplings and time modulation of the middle coupler.

In experiments, however, qubit states are not perfectly localized in the bare product basis; they are better described in the dressed basis of the interacting system. We therefore need absorb static $V_0$ and perform a frame change to redefine both the diagonal spectrum and the time-dependent drive term. In the dispersive limit, one can obtain this transformation using a Schrieffer–Wolff approach. But for more accurate predictions, we instead diagonalize the full static Hamiltonian numerically and re-express the relevant operators in the dressed basis.

With this setup, we now use realistic transmon-coupler parameters to simulate the iSWAP gate.

In [ ]:
w1_num = 3.83 * 2 * np.pi  # GHz
alpha1_num = -0.205 * 2 * np.pi  # GHz
w2_num = 3.11 * 2 * np.pi  # GHz
alpha2_num = -0.216 * 2 * np.pi  # GHz
wc_num = 4.29 * 2 * np.pi  # GHz
alphac_num = -0.161 * 2 * np.pi  # GHz

g1c_num = 0.115 * 2 * np.pi  # GHz
g2c_num = 0.110 * 2 * np.pi  # GHz
g12_num = 0.015 * 2 * np.pi  # GHz

We now substitute the numerical parameter values to construct the corresponding `Qobj` operators.

In [ ]:
H_q1_num = (
    w1_num * a_q1.dag() @ a_q1
    + (alpha1_num / 2) * a_q1.dag() @ a_q1.dag() @ a_q1 @ a_q1
)
H_q2_num = (
    w2_num * a_q2.dag() @ a_q2
    + (alpha2_num / 2) * a_q2.dag() @ a_q2.dag() @ a_q2 @ a_q2
)
H_c_num = (
    wc_num * a_qc.dag() @ a_qc
    + (alphac_num / 2) * a_qc.dag() @ a_qc.dag() @ a_qc @ a_qc
)

V_0_num = (
    g1c_num * ((a_q1 - a_q1.dag()) @ (a_qc - a_qc.dag()))
    + g2c_num * ((a_q2 - a_q2.dag()) @ (a_qc - a_qc.dag()))
    + g12_num * ((a_q1 - a_q1.dag()) @ (a_q2 - a_q2.dag()))
)

H_0_num = H_q1_num + H_q2_num + H_c_num + V_0_num
V_1_num = a_qc.dag() @ a_qc

From this Hamiltonian, we move to the dressed basis by computing its eigenstates and constructing the transformation matrix $U$. In this basis, we extract the diagonal static Hamiltonian, $H_{0,\mathrm{dressed}}$, and the drive-modulation operator, $V_{1,\mathrm{dressed}}$.

In [ ]:
evals, evecs = H_0_num.eigenstates()
sorted_evals, sorted_evecs = SortedFRFSpectrum(evals, evecs, dim_q1, dim_c, dim_q2)
#lexikographisch also |i,j,k> -> Index = i * (dc*dq2) + j * dq2 + k
index_100 = dim_c*dim_q2
index_001 = 1
index_111 = 1*(dim_c*dim_q2)+1*(dim_q2)+1

# transformation matrix from bare basis to dressed basis
U = Qobj(
    np.column_stack(
        [
            sorted_evecs[i, j, k].full()
            for i in range(dim_q1)
            for j in range(dim_c)
            for k in range(dim_q2)
        ]
    ),
    dims=[[dim_q1, dim_c, dim_q2], [dim_q1, dim_c, dim_q2]],
)
H_0_dressed = U.dag() @ H_0_num @ U
V_1_dressed = U.dag() @ V_1_num @ U
display(V_1_dressed)
display(H_0_dressed)

Quantum object: dims=[[3, 3, 3], [3, 3, 3]], shape=(27, 27), type='oper', dtype=Dense, isherm=True
Qobj data =
[[ 4.27708816e-04  6.56895557e-18 -1.76562088e-03 -1.52724660e-18
  -1.38058065e-02  3.82084941e-19  6.53517889e-03 -7.47328870e-19
   9.30338853e-06 -1.32011175e-18 -4.79552276e-03  3.07173707e-19
  -1.15610972e-02 -3.93190362e-19 -6.54477817e-06  2.00805888e-21
  -3.39332661e-06  6.88962890e-21 -5.95234793e-03  6.48240190e-20
  -1.15700995e-06 -2.68915991e-19 -4.41247202e-06 -4.63213938e-21
   8.84452706e-06  5.68111551e-21 -3.53392923e-07]
 [ 6.56895557e-18  9.90287348e-03 -1.59550151e-17  9.31403905e-02
  -3.57144800e-16 -2.13415360e-02  1.39220795e-16 -5.23110921e-03
   3.72458135e-17  2.12411894e-02 -3.60896754e-17 -7.07636844e-03
  -1.45466508e-17  1.22974455e-02 -5.59482364e-17 -2.53338300e-04
   1.09803518e-16 -2.01253536e-06 -7.58265247e-17 -3.33399837e-03
  -9.15874668e-18 -2.31424863e-03  3.42581134e-17 -3.40531926e-06
   7.03224070e-17 -4.68606526e-06  1.90115890e

Quantum object: dims=[[3, 3, 3], [3, 3, 3]], shape=(27, 27), type='oper', dtype=Dense, isherm=True
Qobj data =
[[-2.08856802e-02  2.68868012e-16  2.83676807e-16 -2.38400904e-17
   1.93622982e-15  6.83675470e-18 -1.02506854e-15  2.42315771e-18
  -1.65392541e-17 -1.61682542e-17  1.00693181e-15  3.62154293e-18
   1.56321469e-15 -3.70234806e-18  2.00690851e-17  6.06122385e-19
   4.36754972e-19  1.59573943e-19  8.22815259e-16  1.04142525e-18
   6.26260852e-18  1.52536460e-18  8.98912483e-18  1.14200945e-19
  -2.04732706e-18  9.37690546e-20 -3.99054659e-19]
 [ 2.68868012e-16  1.94377993e+01  8.98679444e-15 -4.27154339e-15
  -5.40648806e-15  3.15580114e-15  7.60498054e-15  3.26829256e-15
   3.72756367e-15 -7.73701535e-15  7.95904391e-16  9.49945701e-16
   1.48338664e-15 -1.33710229e-14 -2.94812358e-15  1.43789407e-15
   5.49770029e-15  3.12627637e-16 -1.40342307e-14  2.91386522e-15
   2.41085335e-15  2.53367627e-15  2.83538591e-15  4.06037258e-16
   1.10228238e-15  7.92619965e-16  2.12624049e

In [ ]:
import numpy as _np

def find_closest_state(energy_value, evals=evals, dims=(dim_q1, dim_c, dim_q2)):
    """
    Find the state |i,j,k> whose energy (in the same units as `evals`) is closest to `energy_value`.
    Returns: (linear_index, (i,j,k), energy, difference)
    """

    arr = evals
    idx = int(_np.argmin(_np.abs(arr - energy_value)))
    i, j, k = _np.unravel_index(idx, dims)
    energy = float(arr[idx])
    diffs = _np.abs(arr - energy_value)
    closest_idxs = _np.argsort(diffs)[:3]

    closest_states = []
    for idx in closest_idxs:
        i, j, k = _np.unravel_index(int(idx), dims)
        energy = float(arr[int(idx)])
        diff = float(energy - energy_value)
        closest_states.append((int(idx), (i, j, k), energy, diff))
        print(f"Closest state: |{i}{j}{k}> (linear index {int(idx)})  energy={energy:.12g}  diff={diff:.12g}")
    return

res_state_freq = 23.87584256065358
find_closest_state(res_state_freq)
evals


Closest state: |002> (linear index 2)  energy=23.8758425607  diff=9.94759830064e-14
Closest state: |010> (linear index 3)  energy=27.1401892664  diff=3.26434670571
Closest state: |001> (linear index 1)  energy=19.4377992742  diff=-4.43804328642


array([-2.08856802e-02,  1.94377993e+01,  2.38758426e+01,  2.71401893e+01,
        3.75958484e+01,  4.33218933e+01,  4.65606558e+01,  4.66144275e+01,
        5.08370003e+01,  5.34814091e+01,  6.14526445e+01,  6.46150185e+01,
        6.60792515e+01,  7.02961671e+01,  7.30471828e+01,  7.33927646e+01,
        7.76594284e+01,  8.41946214e+01,  8.83134176e+01,  9.10808788e+01,
        9.28260627e+01,  9.72724912e+01,  9.99308965e+01,  1.10800550e+02,
        1.15275827e+02,  1.19582303e+02,  1.37549608e+02])

After implementing the dressed state transformation, we extract the energy difference between two states without any drives.

In [ ]:
static_resonant_frequency = (sorted_evals[1, 0, 0] - sorted_evals[0, 0, 1]) / (
    2 * np.pi
)
print(
    "Dressed state difference without drive (GHz): ",
    static_resonant_frequency,
)
energy_d = sorted_evals[1,1,1]
print(energy_d)
print("index_d: ", energy_d)

Dressed state difference without drive (GHz):  0.7063365266895203
70.29616707773066
index_d:  70.29616707773066


Also, the projectors are used to keep track of populations of different states we interested.

In [ ]:
projection_100 = sorted_evecs[1, 0, 0] @ sorted_evecs[1, 0, 0].dag()
projection_010 = sorted_evecs[0, 1, 0] @ sorted_evecs[0, 1, 0].dag()
projection_001 = sorted_evecs[0, 0, 1] @ sorted_evecs[0, 0, 1].dag()
projection_111 = sorted_evecs[1, 1, 1] @ sorted_evecs[1, 1, 1].dag()

## Trying the intuitive approach

Above it was argued that the intuitive idea on driving the coupler at the energy frequency $\omega_d =\omega_1-\omega_2$ does not work (because of amplitude-dependent Stark shifts). But is this really true? Let's do a quick consistency check and sweep the drive frequency vs the drive amplitude:

In [ ]:
lwd = np.linspace(0.700, 0.712, 20) * 2 * np.pi
lA = np.linspace(0.001, 0.2, 20) * 2 * np.pi

mesh_wd, mesh_A = np.meshgrid(lwd, lA, indexing="ij")
final_pop_grid = np.zeros((len(lwd), len(lA)))

tlist = np.linspace(0, 200, 1000)

# for i, wd in enumerate(tqdm(lwd)):
#     for j, A in enumerate(lA):
#         result = mesolve(
#             [H_0_num, [V_1_num, lambda t, args: A*np.cos(wd*t)]],
#             sorted_evecs[1, 0, 0],
#             tlist,
#             [],
#             e_ops=[projection_001],
#         )
#         final_pop_grid[i, j] = result.expect[0][-1]

In [ ]:
# fig, ax = plt.subplots()

# Hauptplot (Population)
# pcm = ax.pcolormesh(
#     mesh_wd / (2 * np.pi),
#     mesh_A / (2 * np.pi),
#     final_pop_grid,
#     shading="auto",
#     rasterized=True,
#     cmap="Blues",
# )

# fig.colorbar(pcm, ax=ax, label="Population in |001>")
# plt.axvline(x = static_resonant_frequency, color = 'red')

# ax.set_xlabel("Drive frequency (GHz)")
# ax.set_ylabel("Drive amplitude (GHz)")
# ax.set_title("Population in |001> after 200 ns starting from |100>")

[**Exercise**: Can you also visualize the dressed state difference in this plot as a comparision to see how far it deviates?]

Oh no: we see indeed that there is a non-trivial resonance condition, that we must implement. Especially for large drive amplitudes (aka fast gates) which we eventually want for stable operations. Let's attack this in the next section.

## Implementing the resonance condition

With the dressed-basis operators in hand, we can now apply the resonance-condition function introduced above.

Before solving for the resonant drive explicitly, we have performed a sweep over drive frequency and amplitude to map the population-transfer landscape. This gives a direct numerical reference for where the strongest transfer occurs.

We then use the resonance-condition function we built before to compute the predicted resonant frequency as a function of amplitude and compare that prediction against the sweep.
To do this, we first convert the `Qobj` object $V_{1,\mathrm{dressed}}$ into a NumPy array.

In [ ]:
E_array = sorted_evals.reshape(dim_q1 * dim_c * dim_q2)
V_1_dressed_array = (
    V_1_dressed.full().reshape(dim_q1 * dim_c * dim_q2, dim_q1 * dim_c * dim_q2).real
)

We take $r=2$ and compute our prediction of the amplitude-dependent resonant drive frequencies

In [ ]:
lresonant_wd_order2 = np.zeros_like(lA, dtype=float)
wd_initial_guess = static_resonant_frequency*2*np.pi
print("guess: ", wd_initial_guess
      )
for j, A in enumerate(tqdm(lA)):
    # sucht nach Nullstelle
    resonant_wd_solution= fsolve(
        resonant_condition, wd_initial_guess, 
        args=(A, 2, state_a, state_b, E_array, None, V_1_dressed_array),
    )
    lresonant_wd_order2[j] = float(resonant_wd_solution[0])
    wd_initial_guess = lresonant_wd_order2[j]
lresonant_wd_order2 /(2*np.pi)

guess:  4.438043286419855


  0%|          | 0/20 [00:00<?, ?it/s]


NameError: name 'state_a' is not defined

We compare the theoretical predictions with the numerical reference.

In [ ]:
fig, ax = plt.subplots()
pcm = ax.pcolormesh(
    mesh_wd / (2 * np.pi),
    mesh_A / (2 * np.pi),
    final_pop_grid,
    shading="auto",
    rasterized=True,
    cmap="Blues",
)
fig.colorbar(pcm, ax=ax, label="Population in |001>")
ax.plot(
    lresonant_wd_order2 / (2 * np.pi),
    lA / (2 * np.pi),
    ls="--",
    color="red",
    label="Resonant frequency (2nd)",
)
ax.set_xlabel("Drive frequency (GHz)")
ax.set_ylabel("Drive amplitude (GHz)")
ax.set_title("Population in |001> after 200 ns starting from |100>")
ax.legend()

We see that implementing the resonance condition helps us to determine the drive frequency even at high drive amplitudes.

## Population Transfer Rate

Having determined the amplitude-dependent drive frequency in the previous section, we can now use it to estimate the Rabi rate. As we want to stop the drive at the point of maximal population transfer (to obtain an iSWAP gate), we want to have very precise predictions of the Rabi rate. As we compute the Rabi rate pertrubatively to order $r$, it is expected that higher orders can give more precise predictions. Let us test numerically, how high we should in perturbation order.

We take $r=2$ and compute our prediction of the Rabi rate, using our previoulsy defined funtion:

[**Excersise**: Let's sweep the rabi rate for all amplitudes in lA. You may use rabi rate function you used before.]

In [ ]:
lresonant_rabi_order2 = np.zeros_like(lA)

for index in range(len(lA)):
    wd = lresonant_wd_order2[index]
    amp = lA[index]
    rabi = rabi_rate(wd, amp, 2, E_array, None, V_1_dressed_array)
    lresonant_rabi_order2[index] = np.abs(rabi)

Using the resonant frequencies predicted by second-order perturbation theory, we numerically compute the final population at the end of the evolution.

If the perturbative description is accurate, the population predicted from the Rabi oscillation (using the perturbative Rabi rate) should agree with the numerical simulation.

In [ ]:
# Evolution along the resonant frequency and drive amplitude
lpop_resonant = np.zeros_like(lA)

tlist = np.linspace(0, 200, 1000)
for i in tqdm(range(len(lA))):
    amp = lA[i]
    fre = lresonant_wd_order2[i]
    result = mesolve(
        [H_0_num, [V_1_num, lambda t, args: amp * np.cos(fre * t)]],
        sorted_evecs[1, 0, 0],
        tlist,
        [],
        e_ops=[projection_001],
    )
    lpop_resonant[i] = result.expect[0][-1]
# display(lresonant_wd_order2/(2*np.pi))
# display(lresonant_rabi_order2)
display("a: ", sorted_evals[1,0,0]/(2*np.pi))
display("b: ", sorted_evals[0,0,1]/(2*np.pi))
display("c: ", sorted_evals[0,1,0]/(2*np.pi))


With resonant states in the effective Hamiltonian, time evolution in the effective Hamiltonian is
$$
P = \sin^2(\Omega_{ab} t)
$$

In [ ]:
fig, ax = plt.subplots()
ax.plot(lA / (2 * np.pi), lpop_resonant, marker="o", label="Numerical Simulations")
ax.plot(
    lA / (2 * np.pi),
    np.sin(np.array(lresonant_rabi_order2) * tlist[-1]) ** 2,
    linestyle="--",
    label="Order 2",
)
ax.set_xlabel("Drive Amplitude (GHz)")
ax.set_ylabel("Population in |001>")
ax.legend()

Can we mitigate the errros in large amplitudes by going to higher order?

(Due to the long computing time in higher orders, we may directly import pre-computed values from saved dataset.)

In [ ]:
ds = xr.open_dataset("resonant_frequency_and_rabi_rate.nc")
lresonant_wd_order3 = ds["resonant_frequency_order_3"].values
lresonant_wd_order4 = ds["resonant_frequency_order_4"].values
lresonant_rabi_order3 = ds["rabi_rate_order_3"].values
lresonant_rabi_order4 = ds["rabi_rate_order_4"].values
ds.close()

In [ ]:
fig, ax = plt.subplots()
ax.plot(lA / (2 * np.pi), lpop_resonant, marker="o", label="Numerical Simulations")
ax.plot(
    lA / (2 * np.pi),
    np.sin(np.array(lresonant_rabi_order2) * tlist[-1]) ** 2,
    linestyle="--",
    label="Order 2",
)
ax.plot(
    lA / (2 * np.pi),
    np.sin(np.array(lresonant_rabi_order3) * tlist[-1]) ** 2,
    linestyle="--",
    label="Order 3",
)
ax.plot(
    lA / (2 * np.pi),
    np.sin(np.array(lresonant_rabi_order4) * tlist[-1]) ** 2,
    linestyle="--",
    label="Order 4",
)
ax.set_xlabel("Drive Amplitude (GHz)")
ax.set_ylabel("Population in |001>")
ax.legend()

# Improving the Fidelities

The solution to the conundrum in the last section is the large leakage to $|010\rangle$ state when going to higher amplitudes. This is ultimately tied to the topic of **Fast Oscillations**.

## Addressing Fast Oscillations

Now we test for the maximal drive amplitude and see the real-time evolutions to figure out what causes the imperfect predictions of the rabi rate.

In [ ]:
test_fre = lresonant_wd_order2[-1]
test_amp = lA[-1]

test_tlist = np.arange(0, 250, 1)
test_dynamics = mesolve(
    [H_0_num, [V_1_num, lambda t, args: test_amp * np.cos(test_fre * t)]],
    sorted_evecs[1, 0, 0],
    test_tlist,
    [],
    e_ops=[projection_001, projection_100, projection_010, projection_111],
    options={"progress_bar": "tqdm"},
)
# display(lresonant_wd_order2/(2*np.pi))
# display(lA/(2*np.pi))
test_amp

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].plot(test_tlist, test_dynamics.expect[0], label="|001>")
ax[0].plot(test_tlist, test_dynamics.expect[1], label="|100>")
ax[0].plot(
    test_tlist,
    np.sin(np.array(lresonant_rabi_order2[-1]) * test_tlist) ** 2,
    ls="--",
    label="2nd order",
    color="C0",
)
ax[0].set_xlabel("Time (ns)")
ax[0].set_ylabel("Population")
ax[0].set_title("Dynamics under resonant drive")
ax[0].legend()

ax[1].plot(test_tlist, test_dynamics.expect[2], label="|010>", color="C2")
ax[1].plot(test_tlist, test_dynamics.expect[3], label="|111>", color="C3")
ax[1].set_ylim(1e-10, 1)
ax[1].set_xlabel("Time (ns)")
ax[1].set_ylabel("Population")
ax[1].set_title("Leakages")
ax[1].set_yscale("log")
ax[1].legend()

In the plot, we saw that the predictions from pure Rabi oscillations (dashed line) deviate from real population transfer (solid lines).

Apart from that, there are two key observations in the figure:

- In the population of $|100\rangle$, there are a lot of wiggles during the time. This can be understood as an anology to the counter-rotating terms in the rotating wave approximations. However, we did not apply the rotating wave approximaton, so these fast oscillations results from hybridizations of intermediate states participating in the eigenstates in Sambe representations.
- In the states not in the computational subspace, there is a large population in the coupler, which is the $|010\rangle$ state. To include the non-negligable population, we need to include the third state to enlarge the degenerate subspace. i.e.,
$$ H_{\text{eff}} =
\begin{array}{c}
|001\rangle \\
|100\rangle \\
|010\rangle
\end{array}
\begin{bmatrix}
\delta_a & \Omega_{ab} & \Omega_{ac} \\
\Omega_{ab}^* & \delta_b & \Omega_{bc} \\
\Omega_{ab}^* & \Omega_{bc}^* & \delta_c \\
\end{bmatrix}
$$

In [ ]:
initial_state = np.zeros(dim_q1 * dim_c * dim_q2)
initial_state[state_a] = 1

We now use built-on function "Psi_t_FloquetPertub" for computing the dynamics predicted in perturbation theory.

In [ ]:
rH = 2
rW = 0
psi_t_rHeff2_rW0, _ = Psi_t_FloquetPerturb_dynamic2(
    rH,
    rW,
    lambda t: test_fre,
    lambda t: test_amp,
    {state_a:1, state_b:0, state_c:2}, # E_state_a - wd = E_state_b = E_state_c - 2*wd
    E_array,
    V_1_dressed_array,
    initial_state,
    test_tlist,
)
resonance_difference = resonant_condition(test_fre, test_amp, 3, state_a, state_b, E_array, None, V_1_dressed_array, analytics = False)
resonance_difference


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].plot(test_tlist, test_dynamics.expect[0], label="|001>")
ax[0].plot(test_tlist, test_dynamics.expect[1], label="|100>")
ax[0].plot(
    test_tlist,
    np.abs(psi_t_rHeff2_rW0[state_a, :]) ** 2,
    "--",
    label="|100> (rH=2, rW=0)",
)
ax[0].set_xlabel("Time (ns)")
ax[0].set_ylabel("Population")
ax[0].set_title("Dynamics under resonant drive")
ax[0].legend()
ax[0].set_ylim(1e-3, 1)

ax[1].plot(test_tlist, test_dynamics.expect[2], label="|010>", color="C1")
ax[1].plot(
    test_tlist,
    np.abs(psi_t_rHeff2_rW0[state_c, :]) ** 2,
    "--",
    label="rH=2, rW=0",
    color="C2",
)
ax[1].set_ylim(1e-5, 1)
ax[1].set_xlabel("Time (ns)")
ax[1].set_ylabel("Population")
ax[1].set_title("Population in |010> under resonant drive")
ax[1].set_yscale("log")
ax[1].legend()

[**Bonus**: You may try to change rW to see how prediced dynamics changes. Why would large rW help?]

By seeing the fast oscillations are captured by including the third state, we now again sweep the drive amplitude in lA to see if the populations in the end of the time are correctly captured.

In [ ]:
lpop_order2 = np.zeros_like(lA)

for i in tqdm(range(len(lA))):
    amp = lA[i]
    fre = lresonant_wd_order2[i]
    psi_t, W_element = Psi_t_FloquetPerturb(
        2,
        0,
        fre,
        amp,
        {state_a: 1, state_b: 0, state_c: 2},  # E_state_a - wd = E_state_b
        E_array,
        V_1_dressed_array,
        initial_state,
        tlist,
    )
    lpop_order2[i] = np.abs(psi_t[state_b, -1]) ** 2

We now benchmark effective models with different choices of states in the degenerate subspace. We use the following notation:
- 2-LS: effective model containing the computational states $|001\rangle$ and $|100\rangle$
- 3-LS: effective model containing $|001\rangle$, $|100\rangle$, and the leakage state $|010\rangle$

Throughout this comparison, full numerical simulation is used as the reference.

In [ ]:
fig, ax = plt.subplots()
ax.plot(lA / (2 * np.pi), lpop_resonant, marker="o", label="Numerical Simulations")
ax.plot(
    lA / (2 * np.pi),
    np.sin(np.array(lresonant_rabi_order2) * tlist[-1]) ** 2,
    linestyle="--",
    label="2-LS Order 2",
)
ax.plot(
    lA / (2 * np.pi),
    lpop_order2,
    marker="x",
    linestyle="--",
    label="3-LS Order 2",
)
ax.set_xlabel("Drive Amplitude (GHz)")
ax.set_ylabel("Population in |001>")
ax.legend()

## Analysing the improved Gate Fidelities

The analytical model predicts both the time evolution and the leakage dynamics well. We now compute the full time evolution directly for comparison.

Using the resonant frequencies predicted by second-order perturbation theory, we need take the fast oscillation into account and extract the gate time for the iSWAP interaction. The way of finding correct pulse length relies on the propator $U$. At each time, we can compute the propagator $U$ from three states involved and evaluate corresponding fidelity with respect to the ideal evolution $U_{\text{ideal}}$:
$$
\text{Fidelity} = \frac{\left|\mathrm{Tr}\left(U_{\text{ideal}}^{\dagger} U\right)\right|^2}{d^2}.
$$
with
$$
U_{\text{ideal}} =
\begin{bmatrix}
0 & i & 0 \\
i & 0 & 0 \\
0 & 0 & 1 \\
\end{bmatrix}
$$
With this in hand, the pulse length is determined by maximal fidelity one can achieve during the time evoluitons.

We use perturbation theory to first find optimal fidelity at second order perturbation theory.

In [ ]:

lt_optimal_prediction = np.zeros_like(lA)
lfidelity_optimal_prediction = np.zeros_like(lA)
lU_optimal_prediction = np.zeros((len(lA), 3, 3), dtype=complex)
ltguess = []
for i in tqdm(range(len(lA))):
    tguess = int(2 * np.pi / lresonant_rabi_order2[i]) / 4
    ltguess.append(tguess)
    optimal_t, optimal_fidelity, optimal_U, fidlist, tlist = find_optimal_time(
        lresonant_wd_order2[i],
        lA[i],
        3,
        state_a,
        state_b,
        state_c,
        E_array,
        V_1_dressed_array,
        tguess,
    )
    lU_optimal_prediction[i] = optimal_U
    lt_optimal_prediction[i] = optimal_t
    lfidelity_optimal_prediction[i] = optimal_fidelity
display(lA/(2*np.pi))
display(lt_optimal_prediction)
display(ltguess)
lfidelity_optimal_prediction

plt.figure()
plt.plot(tlist, fidlist, label='fidelities aggainst gate time')
plt.xlabel('t (ns)')
plt.ylabel('epsilon')
plt.legend()
plt.grid(True)
plt.show()


With the optimal times identified from perturbation theory, we now perform full numerical simulations to compute the exact system evolution and extract the unitary propagator $U(t)$ for each amplitude. This allows us to evaluate the true fidelity without perturbative approximations.

For computational efficiency, we exploit the structure of the problem: rather than evolving the full system, we evolve each of the three computational states separately and stack their final amplitudes to form the 3×3 unitary matrix $U$ restricted to the relevant subspace. The gate fidelity is then computed as:
$$
\text{Fidelity} = \frac{\left|\mathrm{Tr}\left(U_{\text{ideal}}^{\dagger} U\right)\right|^2}{d^2}
$$
where $d=3$ is the dimension of the subspace and $U_{\text{ideal}}$ is the target iSWAP unitary. This comparison reveals the discrepancy between the perturbative predictions and the true dynamics.

In [ ]:
display(lt_optimal_prediction)
display(lresonant_wd_order2)
display(lA)
lfidelity_optimal_numerical = np.zeros_like(lA)
lU_optimal_numerical = np.zeros((len(lA), 3, 3), dtype=complex)

# Define ideal iSWAP gate for comparison
U_ideal = np.array([[0, 1j, 0], [1j, 0, 0], [0, 0, 1]], dtype=complex)

for i in tqdm(range(len(lA))):
    fre = lresonant_wd_order2[i]
    amp = lA[i]

    tlist = np.arange(0, lt_optimal_prediction[i], 0.01)
    print("t_list: ",len(tlist))
    print("end: ", lt_optimal_prediction[i])

    # Instead of computing full propagator, evolve each basis state separately
    # This is much faster for extracting just the 3x3 subsystem
    subsystem_indices = [state_a, state_b, state_c]
    U_subsystem = np.zeros((3, 3), dtype=complex)

    for col_idx, initial_state_idx in enumerate(subsystem_indices):
        # Create initial state |state_idx⟩
        initial_state = np.zeros(len(E_array))
        initial_state[initial_state_idx] = 1
        initial_state = Qobj(initial_state, dims=[[dim_q1, dim_c, dim_q2], [1]])

        # Evolve using mesolve
        result = mesolve(
            [H_0_dressed, [V_1_dressed, lambda t, args: amp * np.cos(fre * t)]],
            initial_state,
            tlist,
            [],
        )
        final_state = result.states[-1]

        # Extract subsystem components to form column of unitary
        for row_idx, j_state in enumerate(subsystem_indices):
            U_subsystem[row_idx, col_idx] = final_state.full()[j_state, 0]

    lU_optimal_numerical[i] = U_subsystem

    # Compute gate fidelity: F = |Tr(U_ideal† U_subsystem)|² / 3²
    trace_overlap = np.trace(U_ideal.conj().T @ U_subsystem)
    lfidelity_optimal_numerical[i] = np.abs(trace_overlap) ** 2 / 3**2

If computing takes too long, it's possible to import the values from pre-computed dataset

In [ ]:
ds = xr.open_dataset("resonant_frequency_and_rabi_rate.nc")
lA = ds["drive_amplitude"].values
lt_optimal_prediction = ds["optimal_time_prediction"].values
lfidelity_optimal_numerical = ds["optimal_fidelity_numerical"].values
lU_optimal_numerical = ds["optimal_U_numerical"].values
ds.close()

To understand the sources of infidelity, we decompose the total error into two components: **leakage error** and **remaining error**. Leakage occurs when the quantum state escapes the computational subspace spanned by $|001\rangle$ and $|100\rangle$ into the third state such as $|010\rangle$.

For each unitary $U$ obtained from the numerical evolution, the leakage error is quantified by the loss of probability density in the computational 2×2 block:
$$
\text{Leakage} = 1 - \frac{\mathrm{Tr}(U_{\text{comp}}^{\dagger} U_{\text{comp}})}{2}
$$
where $U_{\text{comp}}$ is the $2 \times 2$ restriction of $U$ to the computational subspace. The remaining error is attributed to phase shifts and incomplete rotation within the computational subspace.

In [ ]:
lleakage_error_numerical = np.zeros_like(lA)

for i in tqdm(range(len(lA))):
    U = lU_optimal_numerical[i]
    leakage_numerical = 1 - np.real(np.trace(U[:2, :2].conj().T @ U[:2, :2])) / 2
    lleakage_error_numerical[i] = leakage_numerical

In [ ]:
# compare the optimal predictions with numerical simulations
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax.plot(
    lA / (2 * np.pi),
    1 - lfidelity_optimal_numerical,
    "o-",
    label="Total Error",
    color="C0",
)
ax.plot(
    lA / (2 * np.pi), lleakage_error_numerical, "o--", label="Leakage Error", color="C0"
)
ax.set_xlabel("Drive Amplitude (GHz)")
ax.set_ylabel("Total Error")
ax.set_title("Analysis of Errors")
ax.set_yscale("log")
ax.legend()

From this comparison, it is clear that leakage becomes dominant as the drive amplitude increases. To suppress this leakage and recover high fidelity, one can use optimized pulse shaping, such as DRAG pulses. Another important error source is phase accumulation during the pulse. Because the population transfer is not a perfect Rabi oscillation, an additional control degree of freedom—the drive phase—is needed to compensate these phase errors.

Further explorative tasks:
- How to further improve the fidelities?
- Leakages in other states, i.e., $|000\rangle$ and $|101\rangle$
- Extending to bSWAP gate, i.e., $|000\rangle \leftrightarrow |101\rangle$ or C-Phase gate, i.e., $|101\rangle \leftrightarrow |200\rangle$.

# Save Data

In [ ]:
ds = xr.Dataset(
    {
        "resonant_frequency_order_2": (("drive_amplitude"), lresonant_wd_order3),
        "resonant_frequency_order_3": (("drive_amplitude"), lresonant_wd_order3),
        "resonant_frequency_order_4": (("drive_amplitude"), lresonant_wd_order4),
        "rabi_rate_order_2": (("drive_amplitude"), lresonant_rabi_order2),
        "rabi_rate_order_3": (("drive_amplitude"), lresonant_rabi_order3),
        "rabi_rate_order_4": (("drive_amplitude"), lresonant_rabi_order4),
        "optimal_time_prediction": (("drive_amplitude"), lt_optimal_prediction),
        "optimal_fidelity_prediction": (
            ("drive_amplitude"),
            lfidelity_optimal_prediction,
        ),
        "optimal_U_prediction": (
            ("drive_amplitude", "subspace_row", "subspace_col"),
            lU_optimal_prediction,
        ),
        "optimal_fidelity_numerical": (
            ("drive_amplitude"),
            lfidelity_optimal_numerical,
        ),
        "optimal_U_numerical": (
            ("drive_amplitude", "subspace_row", "subspace_col"),
            lU_optimal_numerical,
        ),
    },
    coords={
        "drive_amplitude": lA,
        "subspace_row": np.arange(3),
        "subspace_col": np.arange(3),
    },
)
ds.to_netcdf("resonant_frequency_and_rabi_rate.nc", auto_complex=True)

In [ ]:
order = 3
lresonant_wd_order3 = np.zeros_like(lA)
lresonant_rabi_order3 = np.zeros_like(lA)

for j, A in enumerate(tqdm(lA)):
    wd_initial_guess = 2 * np.pi * 0.705
    resonant_wd_solution = fsolve(
        resonant_condition,
        wd_initial_guess,
        args=(A, order, state_a, state_b, E_array, None, V_1_dressed_array),
    )
    lresonant_wd_order3[j] = resonant_wd_solution[0]

    rabi = rabi_rate(
        resonant_wd_solution[0],
        A,
        order,
        state_b,
        state_a,
        E_array,
        V0=None,
        V1=V_1_dressed_array,
    )
    lresonant_rabi_order3[j] = np.abs(rabi)

In [ ]:
order = 4
lresonant_wd_order4 = np.zeros_like(lA)
lresonant_rabi_order4 = np.zeros_like(lA)

for j, A in enumerate(tqdm(lA)):
    wd_initial_guess = 2 * np.pi * 0.705
    resonant_wd_solution = fsolve(
        resonant_condition,
        wd_initial_guess,
        args=(A, order, state_a, state_b, E_array, None, V_1_dressed_array),
    )
    lresonant_wd_order4[j] = resonant_wd_solution[0]

    rabi = rabi_rate(
        resonant_wd_solution[0],
        A,
        order,
        state_b,
        state_a,
        E_array,
        V0=None,
        V1=V_1_dressed_array,
    )
    lresonant_rabi_order4[j] = np.abs(rabi)
    display(resonant_wd_solution)
    lresonant_rabi_order4

In [ ]:
lleakage1 = np.zeros_like(lA)
lleakage2 = np.zeros_like(lA)

for i in tqdm(range(len(lA))):
    fre = lresonant_wd_order2[i]
    amp = lA[i]

    matrix = Heff_Floquet_matrix_summed(
        2,
        fre,
        {state_a: 1, state_b: 0, state_c: 2},
        E_num,
        amp / 2 * V1_num,
        V0=None,
    )
    lleakage1[i] = np.abs(matrix[0, 2])
    lleakage2[i] = np.abs(matrix[1, 2])

In [ ]:
fig, ax = plt.subplots()
ax.plot(lA / (2 * np.pi), lleakage1, label="Leakage to |010> from |001>")
ax.plot(lA / (2 * np.pi), lleakage2, label="Leakage to |010> from |100>")
ax.set_xlabel("Drive Amplitude (GHz)")
ax.set_ylabel("Leakage Matrix Element")
ax.legend()